In [15]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

In [2]:
print(customers_df)

   customerID                         companyName              contactName  \
0       ALFKI                 Alfreds Futterkiste             Maria Anders   
1       ANATR  Ana Trujillo Emparedados y helados             Ana Trujillo   
2       ANTON             Antonio Moreno Taquería           Antonio Moreno   
3       AROUT                     Around the Horn             Thomas Hardy   
4       BERGS                  Berglunds snabbköp       Christina Berglund   
..        ...                                 ...                      ...   
86      WARTH                      Wartian Herkku         Pirkko Koskitalo   
87      WELLI              Wellington Importadora            Paula Parente   
88      WHITC                White Clover Markets           Karl Jablonski   
89      WILMK                         Wilman Kala          Matti Karttunen   
90      WOLZA                      Wolski  Zajazd  Zbyszek Piestrzeniewicz   

                 contactTitle                        address   

In [29]:
print(orders_df)

     orderID customerID  employeeID                orderDate  \
0      10248      VINET           5  1996-07-04 00:00:00.000   
1      10249      TOMSP           6  1996-07-05 00:00:00.000   
2      10250      HANAR           4  1996-07-08 00:00:00.000   
3      10251      VICTE           3  1996-07-08 00:00:00.000   
4      10252      SUPRD           4  1996-07-09 00:00:00.000   
..       ...        ...         ...                      ...   
825    11073      PERIC           2  1998-05-05 00:00:00.000   
826    11074      SIMOB           7  1998-05-06 00:00:00.000   
827    11075      RICSU           8  1998-05-06 00:00:00.000   
828    11076      BONAP           4  1998-05-06 00:00:00.000   
829    11077      RATTC           1  1998-05-06 00:00:00.000   

                requiredDate              shippedDate  shipVia  freight  \
0    1996-08-01 00:00:00.000  1996-07-16 00:00:00.000        3    32.38   
1    1996-08-16 00:00:00.000  1996-07-10 00:00:00.000        1    11.61   
2    1

Task 1 — Aggregation and Grouping

Using only the orders table, write a SQL query that returns each CustomerID along with:

The total number of orders they placed (order_count)
The total freight amount across all their orders (total_freight)
The average freight amount per order (avg_freight)
Sort the results by total_freight in descending order.

Run the query using pd.read_sql_query() and display the top 10 rows.

In [10]:
sql_query = "select customerId,count(orderID) as order_count, sum(freight) as total_freight , avg(freight) as avg_freight from Orders group by (customerId) order by total_freight desc limit 10"

result_df = pd.read_sql_query(sql_query,conn)
print(result_df)

  customerID  order_count  total_freight  avg_freight
0      SAVEA           31        6683.70   215.603226
1      ERNSH           30        6205.39   206.846333
2      QUICK           28        5605.63   200.201071
3      HUNGO           19        2755.24   145.012632
4      RATTC           18        2134.21   118.567222
5      QUEEN           13        1982.70   152.515385
6      FOLKO           19        1678.08    88.320000
7      BERGS           18        1559.52    86.640000
8      FRANK           15        1403.44    93.562667
9      MEREP           13        1394.22   107.247692


Task 2 — WHERE vs. HAVING

Write two separate SQL queries to demonstrate the difference between WHERE and HAVING:

Query A: From the orders table, filter rows where Freight is greater than 50 before aggregation, then group by CustomerID and return the count of such orders as high_freight_orders.

Query B: From the orders table, group by CustomerID and return only those customers whose total freight exceeds 500, using HAVING. Return CustomerID and total_freight.

In a markdown cell below your queries, write 2–3 sentences explaining why Query A and Query B produce different results even though both involve a threshold on Freight.



In [12]:
sql_qiery = "select count(*) as high_freight_orders from orders where freight >50 group by (customerID)"
res = pd.read_sql_query(sql_qiery, conn)

print(res)
print("\n")
sql_qiery2 = "select customerID, sum(freight) as total_freight from orders  group by (customerID) having freight>500"
res2 = pd.read_sql_query(sql_qiery2, conn)

print(res2)

    high_freight_orders
0                     2
1                     2
2                     2
3                    11
4                     1
..                  ...
69                    2
70                    6
71                    1
72                    7
73                    1

[74 rows x 1 columns]


  customerID  total_freight
0      QUEEN         1982.7


Query A: WHERE before aggregation, applies filter on each row.Query A only considers orders where Freight is greater than 50 and then counts them per customer

Query B: Having after Aggregation, applies filter on the grouped result. It first calculates the total freight per customer and then keeps only those customers whose total exceeds 500

Task 3 — JOIN and Aggregation

Write a SQL query that joins the customers and orders tables on CustomerID and returns:

CompanyName (from customers)
Country (from customers)
Total number of orders placed (order_count)
Total freight (total_freight)
Include only customers who have placed at least one order (i.e., use INNER JOIN).

Then write a second query using LEFT JOIN that includes all customers, even those with no orders. For customers with no orders, total_freight should appear as NULL or 0.

Display both results and in a markdown cell, explain in 2–3 sentences what changed between the two queries and why.

In [36]:
sql_query1 = "select c.companyName, c.city, count(o.orderId) as Order_count,sum(o.freight) as Total_freight from customers as c inner join orders as o on c.customerID = o.customerID group by c.customerId having count(c.customerId) >=1"
results = pd.read_sql_query(sql_query1, conn)

sql_query2 = "select c.companyName, c.city, count(o.orderId) as Order_count,sum(o.freight) as Total_freight from customers as c left join orders as o on c.customerID = o.customerID group by c.customerId"
results2 = pd.read_sql_query(sql_query2, conn)

print (results)
print (results2)
print(results2['Total_freight'].isnull().sum())


                           companyName         city  Order_count  \
0                  Alfreds Futterkiste       Berlin            6   
1   Ana Trujillo Emparedados y helados  México D.F.            4   
2              Antonio Moreno Taquería  México D.F.            7   
3                      Around the Horn       London           13   
4                   Berglunds snabbköp        Luleå           18   
..                                 ...          ...          ...   
84                      Wartian Herkku         Oulu           15   
85              Wellington Importadora      Resende            9   
86                White Clover Markets      Seattle           14   
87                         Wilman Kala     Helsinki            7   
88                      Wolski  Zajazd     Warszawa            7   

    Total_freight  
0          225.58  
1           97.42  
2          268.52  
3          471.95  
4         1559.52  
..            ...  
84         822.48  
85         194.71  
86 

In [40]:
#inner join using merge
result1 = pd.merge(customers_df, orders_df, on='customerID', how='inner')

#groupby customerID, applying aggregate
#Here companyName/City "first" --> consider the unique value per customer

final_result1 =(result1
               .groupby("customerID")
               .aggregate(
                   CompanyName = ("companyName","first"),
                   City = ("city","first"),
                   Order_count = ("orderID","size"),
                   Total_Freight = ("freight","sum")
               ).reset_index()
               ) [["CompanyName","City","Order_count","Total_Freight"]]
print(final_result1)

print(final_result1['Total_Freight'].notnull().sum())


                           CompanyName         City  Order_count  \
0                  Alfreds Futterkiste       Berlin            6   
1   Ana Trujillo Emparedados y helados  México D.F.            4   
2              Antonio Moreno Taquería  México D.F.            7   
3                      Around the Horn       London           13   
4                   Berglunds snabbköp        Luleå           18   
..                                 ...          ...          ...   
84                      Wartian Herkku         Oulu           15   
85              Wellington Importadora      Resende            9   
86                White Clover Markets      Seattle           14   
87                         Wilman Kala     Helsinki            7   
88                      Wolski  Zajazd     Warszawa            7   

    Total_Freight  
0          225.58  
1           97.42  
2          268.52  
3          471.95  
4         1559.52  
..            ...  
84         822.48  
85         194.71  
86 

In [41]:
#left join using merge
result1 = pd.merge(customers_df, orders_df, on='customerID', how='left')

#groupby customerID, applying aggregate
#Here companyName/City "first" --> consider the unique value per customer

final_result1 =(result1
               .groupby("customerID")
               .aggregate(
                   CompanyName = ("companyName","first"),
                   City = ("city","first"),
                   Order_count = ("orderID","size"),
                   Total_Freight = ("freight","sum")
               ).reset_index()
               ) [["CompanyName","City","Order_count","Total_Freight"]]
print(final_result1)

#checking for Null
print(final_result1['Total_Freight'].notnull().sum())

                           CompanyName         City  Order_count  \
0                  Alfreds Futterkiste       Berlin            6   
1   Ana Trujillo Emparedados y helados  México D.F.            4   
2              Antonio Moreno Taquería  México D.F.            7   
3                      Around the Horn       London           13   
4                   Berglunds snabbköp        Luleå           18   
..                                 ...          ...          ...   
86                      Wartian Herkku         Oulu           15   
87              Wellington Importadora      Resende            9   
88                White Clover Markets      Seattle           14   
89                         Wilman Kala     Helsinki            7   
90                      Wolski  Zajazd     Warszawa            7   

    Total_Freight  
0          225.58  
1           97.42  
2          268.52  
3          471.95  
4         1559.52  
..            ...  
86         822.48  
87         194.71  
88 

There are two customer who didn't place an order. On left Join, we are getitng cusotmer data who didn't place the order. Whereas in inner join that customer is ignored (common data between tables is fetched)